In [1]:
# IMPORTSSSS
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, GlobalAveragePooling1D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, GlobalAveragePooling1D, Dense, Dropout, Concatenate, RepeatVector, TimeDistributed

In [3]:

# Preprocess blocks
def analyze_stroke_data(csv_filepath):
    # 1. Load the data
    df = pd.read_csv(csv_filepath)
    
    # 2. Calculate time differences between rows (Delta Time / dt)
    df['dt'] = df['time'].diff().fillna(0)
    
    # 3. Calculate distance between points (Delta Distance using Pythagorean theorem)
    df['dx'] = df['x'].diff().fillna(0)
    df['dy'] = df['y'].diff().fillna(0)
    
    # 4. Calculate Velocity (Distance / Time)
    # np.where prevents division-by-zero errors if two events fire at the exact same millisecond
    df['distance'] = np.sqrt(df['dx']**2 + df['dy']**2) 
    df['velocity'] = np.where(df['dt'] > 0, df['distance'] / df['dt'], 0)
    
    # Calculate "Writing Duration" 
    # Sum of the 'dt' column, but ONLY for rows where 'touching' == 1
    writing_duration = df['dt'].where(df['touching'] == 1, 0).sum()

    # Calculate "In-Air Pen Duration" (The pause time biomarker)
    # Sum of the 'dt' column, but ONLY for rows where 'touching' == 0
    in_air_duration = df['dt'].where(df['touching'] == 0, 0).sum()
    
    # TODO 3: Print the results!
    print(f"--- Analysis for: {csv_filepath} ---")
    print(f"Total Writing Duration : {writing_duration} ms")
    print(f"Total In-Air Pauses    : {in_air_duration} ms")
    print(f"Average Pen Velocity   : {df['velocity'].mean():.2f} px/ms")
    print("-" * 40)
    
    return df


In [4]:

def load_and_pad_data(data_dir):
    sequences = []
    latencies = []
    labels = []
    
    for filename in os.listdir(data_dir):
        if not filename.endswith('.csv'):
            continue
            
        # Grab the label (1 for dyslexia, 0 for normal)
        label = 1 if filename.startswith("dyslexia") else 0
        filepath = os.path.join(data_dir, filename)
       
        # loads the CSV and calculates the velocity/distances.
        df = analyze_stroke_data(filepath)
        
        if 'latency' in df.columns:
            latency_val = df['latency'].iloc[0]
        else:
            latency_val = 0
            
        # Extract just the features we want
        stroke_data = df[['velocity', 'pressure', 'touching']].values
        
        # Pad or Truncate to MAX_TIMESTEPS (500)
        if len(stroke_data) > MAX_TIMESTEPS:
            stroke_data = stroke_data[:MAX_TIMESTEPS] # Truncate if too long
        else:
            # Pad with zeros if too short
            padding = np.zeros((MAX_TIMESTEPS - len(stroke_data), FEATURES))
            stroke_data = np.vstack((stroke_data, padding))
            
        sequences.append(stroke_data)
        latencies.append(latency_val)
        # labels.append(label)
        labels.append(np.full((MAX_TIMESTEPS, 1), label))
        
    return np.array(sequences), np.array(latencies), np.array(labels)


In [7]:
# Hyperparameters
MAX_TIMESTEPS = 500  #  standardize all writing samples to 500 time-steps
FEATURES = 3         #  feed it with 3 features: [velocity, pressure, touching]

def build_model():
    sequence_input = Input(shape=(MAX_TIMESTEPS, FEATURES), name="kinematics")
    # padding='same' ensures the sequence stays exactly 500 steps long
    x = Conv1D(32, 3, activation='relu', padding='same')(sequence_input)
    x = Conv1D(64, 3, activation='relu', padding='same')(x)
    # Latency INput
    latency_input = Input(shape=(1,), name="latency")
    # Stretch the 1 single latency number into 500 copies
    repeated_latency = RepeatVector(MAX_TIMESTEPS)(latency_input) 
    
    merged = Concatenate()([x, repeated_latency]) 
    
    y = TimeDistributed(Dense(64, activation='relu'))(merged)
    y = Dropout(0.5)(y)
    final_output = TimeDistributed(Dense(1, activation='sigmoid'))(y)
    
    model = Model(inputs=[sequence_input, latency_input], outputs=final_output)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [8]:
model = build_model()
model.summary()
print("Loading data...")

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ kinematics          │ (None, 500, 3)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 500, 32)   │        320 │ kinematics[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ latency             │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 500, 64)   │      6,208 │ conv1d[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ repeat_vector       │ (None, 500, 1)    │          0 │ latency[0][0]     │
│ (RepeatVector)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 500, 65)   │          0 │ conv1d_1[0][0],   │
│ (Concatenate)       │                   │            │ repeat_vector[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed    │ (None, 500, 64)   │      4,224 │ concatenate[0][0] │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 500, 64)   │          0 │ time_distributed… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_1  │ (None, 500, 1)    │         65 │ dropout[0][0]     │
│ (TimeDistributed)   │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 10,817 (42.25 KB)

 Trainable params: 10,817 (42.25 KB)

 Non-trainable params: 0 (0.00 B)

Loading data...


In [ ]:
# Point this to your data collector folder
X_seq, X_lat, y = load_and_pad_data("../datasets/") 
print(f"Data loaded! \nShape of X_seq (timeseries sequence) : {X_seq.shape} \nShape of X_lat (latency) :{X_lat.shape} \nShape of y (labels) : {y.shape}")
    

In [ ]:
print("Starting training...")
# history = model.fit(x_train, y_train, epochs=20, validation_split=0.2)
history = model.fit([X_seq, X_lat], y, epochs=20, validation_split=0.2)
print("Saving the model...")
model.save("../models/elkinematic-with-lat.keras")